# 06 — Hybrid Gated-Fusion Training
Trains the DistilBERT + XGBoost leaf-encoding hybrid model and saves all artifacts to `models/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import xgboost as xgb
from datasets import load_dataset

from src.text_pipeline import TextEncoder
from src.fusion_model import HybridSalesPredictor

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODELS_DIR = ROOT / 'models'
METRICS_DIR = ROOT / 'metrics'
FIGURES_DIR = ROOT / 'figures'
for d in [MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_SIZE = 5000   # Set to 100000 for full run
BATCH_SIZE  = 32
REPR_DIM    = 128

print(f'Device: {DEVICE}  |  Sample size: {SAMPLE_SIZE}')

## 1 — Load dataset

In [ ]:
print('[1/8] Loading dataset...')
ds = load_dataset('DeepMostInnovations/saas-sales-conversations', split='train')
df = ds.to_pandas()
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
print(f'  Rows: {len(df):,}  Columns: {len(df.columns):,}')
print(f'  Outcome distribution:\n{df["outcome"].value_counts()}')

## 2 — Tabular features

In [ ]:
print('[2/8] Engineering tabular features...')

# Numeric columns
num_cols = [c for c in ['conversation_length','customer_engagement','sales_effectiveness'] if c in df.columns]

# Categorical OHE
cat_cols = [c for c in ['product_type','conversation_style','conversation_flow','communication_channel'] if c in df.columns]

tab_df = df[num_cols + cat_cols].copy()

# Engineered ratios
if 'customer_engagement' in tab_df and 'sales_effectiveness' in tab_df:
    tab_df['eng_eff_ratio'] = tab_df['customer_engagement'] / (tab_df['sales_effectiveness'] + 0.001)
if 'customer_engagement' in tab_df and 'conversation_length' in tab_df:
    tab_df['eng_per_turn'] = tab_df['customer_engagement'] / (tab_df['conversation_length'] + 1)
    tab_df['is_short'] = (tab_df['conversation_length'] < 5).astype(int)
    tab_df['is_long']  = (tab_df['conversation_length'] > 15).astype(int)

final_num_cols = [c for c in tab_df.columns if c not in cat_cols]
scaler = StandardScaler()
X_num = scaler.fit_transform(tab_df[final_num_cols].fillna(0).values)

if cat_cols:
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_cat = enc.fit_transform(tab_df[cat_cols].fillna('unknown').astype(str))
    X_tab = np.hstack([X_num, X_cat])
    cat_feature_names = enc.get_feature_names_out(cat_cols).tolist()
else:
    enc = None
    X_tab = X_num
    cat_feature_names = []

feature_names = final_num_cols + cat_feature_names
y = df['outcome'].astype(int).values
print(f'  Tabular shape: {X_tab.shape}  |  Feature names: {len(feature_names)}')

## 3 — Train XGBoost & extract leaf encodings

In [ ]:
print('[3/8] Training XGBoost + leaf encoding...')

xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=SEED, verbosity=0,
)
xgb_model.fit(X_tab, y)
xgb_model.save_model(str(MODELS_DIR / 'xgboost_model.json'))

# Leaf indices: [N, n_trees]
leaf_indices = xgb_model.get_booster().predict(xgb.DMatrix(X_tab), pred_leaf=True)
print(f'  Leaf index matrix: {leaf_indices.shape}')

# One-hot encode leaves
leaf_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
leaf_ohe = leaf_enc.fit_transform(leaf_indices)
LEAF_DIM = leaf_ohe.shape[1]
print(f'  Leaf OHE shape: {leaf_ohe.shape}  (LEAF_DIM={LEAF_DIM})')

## 4 — Tokenize conversations

In [ ]:
print('[4/8] Tokenizing conversations with DistilBERT...')

text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
texts = df[text_col].fillna('').astype(str).tolist()

from transformers import DistilBertTokenizerFast
tok = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

MAX_LEN = 256  # reduced for CPU speed; use 512 for GPU
CHUNK = 500
all_ids, all_masks = [], []

for i in range(0, len(texts), CHUNK):
    batch = texts[i:i+CHUNK]
    enc = tok(batch, max_length=MAX_LEN, padding='max_length',
               truncation=True, return_tensors='pt')
    all_ids.append(enc['input_ids'])
    all_masks.append(enc['attention_mask'])
    if (i // CHUNK) % 5 == 0:
        print(f'  Tokenized {min(i+CHUNK, len(texts)):,}/{len(texts):,}')

input_ids   = torch.cat(all_ids,   dim=0)
attn_masks  = torch.cat(all_masks, dim=0)
tok.save_pretrained(str(MODELS_DIR / 'tokenizer'))
print(f'  Token tensors: {input_ids.shape}  Tokenizer saved.')

## 5 — Data splits & DataLoaders

In [ ]:
print('[5/8] Splitting data and building DataLoaders...')

idx = np.arange(len(y))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)

leaf_t = torch.tensor(leaf_ohe, dtype=torch.float32)
y_t    = torch.tensor(y, dtype=torch.float32)

def make_loader(indices, shuffle):
    ds = TensorDataset(
        leaf_t[indices], input_ids[indices], attn_masks[indices], y_t[indices]
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

train_loader = make_loader(tr_idx, True)
val_loader   = make_loader(va_idx, False)
print(f'  Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 6 — Build model & train

In [ ]:
print('[6/8] Building Hybrid model...')

text_enc = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
model = HybridSalesPredictor(text_encoder=text_enc, repr_dim=REPR_DIM)
model.set_tab_projection(leaf_dim=LEAF_DIM)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total params: {total:,}  Trainable: {trainable:,}')

criterion = nn.BCELoss()
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

def run_epoch(loader, optimizer, train):
    model.train(train)
    total_loss, preds, probs, labels = 0.0, [], [], []
    with torch.set_grad_enabled(train):
        for leaf_b, ids_b, mask_b, y_b in loader:
            leaf_b, ids_b, mask_b, y_b = (
                leaf_b.to(DEVICE), ids_b.to(DEVICE),
                mask_b.to(DEVICE), y_b.to(DEVICE)
            )
            prob, gate, attn = model(leaf_b, ids_b, mask_b)
            prob = prob.squeeze(-1)
            loss = criterion(prob, y_b)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            probs.extend(prob.detach().cpu().numpy())
            preds.extend((prob.detach().cpu().numpy() > 0.5).astype(int))
            labels.extend(y_b.cpu().numpy())
    n = len(loader)
    return total_loss/n, np.array(preds), np.array(probs), np.array(labels)

# Phase 1: Frozen BERT
print('\n--- Phase 1: Frozen BERT (5 epochs) ---')
opt1 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3
)
for ep in range(5):
    t0 = time.time()
    tr_loss, _, _, _ = run_epoch(train_loader, opt1, True)
    va_loss, va_preds, va_probs, va_labels = run_epoch(val_loader, opt1, False)
    f1  = f1_score(va_labels, va_preds, zero_division=0)
    auc = roc_auc_score(va_labels, va_probs) if len(np.unique(va_labels)) > 1 else 0.0
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_f1'].append(f1)
    history['val_auc'].append(auc)
    print(f'  Ep {ep+1}/5 | tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} f1={f1:.4f} auc={auc:.4f} [{time.time()-t0:.0f}s]')

# Phase 2: Fine-tune last 2 BERT layers
print('\n--- Phase 2: BERT fine-tune (3 epochs) ---')
model.text_encoder.unfreeze_last_n_layers(2)
trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Trainable after unfreeze: {trainable2:,}')

bert_params  = [p for n, p in model.named_parameters() if 'bert' in n and p.requires_grad]
other_params = [p for n, p in model.named_parameters() if 'bert' not in n and p.requires_grad]
opt2 = torch.optim.Adam([{'params': bert_params, 'lr': 2e-5},
                          {'params': other_params, 'lr': 1e-3}])

best_f1, best_state = 0.0, None
for ep in range(3):
    t0 = time.time()
    tr_loss, _, _, _ = run_epoch(train_loader, opt2, True)
    va_loss, va_preds, va_probs, va_labels = run_epoch(val_loader, opt2, False)
    f1  = f1_score(va_labels, va_preds, zero_division=0)
    auc = roc_auc_score(va_labels, va_probs) if len(np.unique(va_labels)) > 1 else 0.0
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_f1'].append(f1)
    history['val_auc'].append(auc)
    print(f'  Ep {ep+1}/3 | tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} f1={f1:.4f} auc={auc:.4f} [{time.time()-t0:.0f}s]')
    if f1 >= best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state:
    model.load_state_dict(best_state)
print(f'\nBest val F1: {best_f1:.4f}')

## 7 — Save artifacts

In [ ]:
print('[7/8] Saving model artifacts...')

torch.save(model, str(MODELS_DIR / 'hybrid_model.pt'))

feature_config = {
    'tabular_features': feature_names,
    'num_cols': final_num_cols,
    'cat_cols': cat_cols,
    'leaf_dim': LEAF_DIM,
    'repr_dim': REPR_DIM,
    'max_len': MAX_LEN,
    'device': DEVICE,
    'sample_size': SAMPLE_SIZE,
    'val_f1': best_f1,
}
with open(MODELS_DIR / 'feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

with open(METRICS_DIR / 'hybrid_training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('  Saved: hybrid_model.pt, xgboost_model.json, tokenizer/, feature_config.json')

## 8 — Plot training curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[0].axvline(5.5, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')

axes[1].plot(epochs, history['val_f1'],  'g-o', ms=4, label='Val F1')
axes[1].set_title('Val F1'); axes[1].grid(True)

axes[2].plot(epochs, history['val_auc'], 'm-o', ms=4, label='Val AUC')
axes[2].set_title('Val AUC'); axes[2].grid(True)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'hybrid_training_curves.png'), dpi=150)
plt.show()
print(f'Final val F1={history["val_f1"][-1]:.4f}  AUC={history["val_auc"][-1]:.4f}')
print('Done ✅')